In [1]:
'''
Steps
1. Data shifting (balance data if imblance by downsampling)
2. Data augmentation
--a) do only Gaussian noise
--b) do only Flipping/mirroring
--c) do only Rotation
--d) do only Normalizing
--e) do all augmentation techniques at the same time
3. Go back to step 1 or 2 and repeat (for data-centric approach)
4. Optimizer
5. Hyperparameter tuning (w/o hyper parameter tuning) and model training and testing with the best dataset
'''
pass

In [2]:
from sklearn import metrics
from tqdm import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision.transforms as transforms
from dataCentric_functions import *

import medmnist
from medmnist import INFO, Evaluator

In [3]:
from libauc.losses import AUCMLoss
from libauc.optimizers import PESG, Adam, SGD
from libauc.models import resnet20 as ResNet20
from libauc.datasets import CIFAR10
from libauc.utils import ImbalancedDataGenerator
from libauc.sampler import DualSampler
from libauc.metrics import auc_roc_score
from libauc.models import resnet18 as ResNet18

import torch 
from PIL import Image
import numpy as np
import torchvision.transforms as transforms
from torch.utils.data import Dataset
from sklearn.metrics import roc_auc_score
#from pytorchtools import EarlyStopping
import sys


In [4]:
import matplotlib.pyplot as plt

In [5]:
import os

# make directory for adrenal
if not os.path.exists('chest'):
    os.makedirs('chest')

In [6]:
#data_flag = 'pathmnist'
data_flag = 'chestmnist'
download = True

NUM_EPOCHS = 3
BATCH_SIZE = 128
lr = 0.001

info = INFO[data_flag]
task = info['task']
n_channels = info['n_channels']
n_classes = len(info['label'])

DataClass = getattr(medmnist, info['python_class'])

In [7]:
# preprocessing
data_transform1 = transforms.Compose([
    transforms.ToTensor()
])


In [8]:
train_dataset1 = DataClass(split='train', transform=data_transform1, download=download)
val_dataset1 = DataClass(split='val', transform=data_transform1, download=download)
test_dataset = DataClass(split='test', transform=data_transform1, download=download)

train_dataset1

Using downloaded and verified file: /Users/chihoonlee/.medmnist/chestmnist.npz
Using downloaded and verified file: /Users/chihoonlee/.medmnist/chestmnist.npz
Using downloaded and verified file: /Users/chihoonlee/.medmnist/chestmnist.npz


Dataset ChestMNIST (chestmnist)
    Number of datapoints: 78468
    Root location: /Users/chihoonlee/.medmnist
    Split: train
    Task: multi-label, binary-class
    Number of channels: 1
    Meaning of labels: {'0': 'atelectasis', '1': 'cardiomegaly', '2': 'effusion', '3': 'infiltration', '4': 'mass', '5': 'nodule', '6': 'pneumonia', '7': 'pneumothorax', '8': 'consolidation', '9': 'edema', '10': 'emphysema', '11': 'fibrosis', '12': 'pleural', '13': 'hernia'}
    Number of samples: {'train': 78468, 'val': 11219, 'test': 22433}
    Description: The ChestMNIST is based on the NIH-ChestXray14 dataset, a dataset comprising 112,120 frontal-view X-Ray images of 30,805 unique patients with the text-mined 14 disease labels, which could be formulized as a multi-label binary-class classification task. We use the official data split, and resize the source images of 1×1024×1024 into 1×28×28.
    License: CC BY 4.0

In [9]:
train_counts = np.unique(train_dataset1.labels, return_counts=True)
val_counts = np.unique(val_dataset1.labels, return_counts=True)
print(f"class ratio: {train_counts}")
print(f"class ratio: {val_counts}")
print(f"class ratio: {np.unique(test_dataset.labels, return_counts=True)}")
train_val_comb = train_counts[1] + val_counts[1]
#neg_counts, pos_counts = train_val_comb[0], train_val_comb[1]
#print(f"train+validation class ratio: {neg_counts, pos_counts}")
print(f"train+validation class ratio: {train_val_comb}")

class ratio: (array([0, 1], dtype=uint8), array([1042052,   56500]))
class ratio: (array([0, 1], dtype=uint8), array([149088,   7978]))
class ratio: (array([0, 1], dtype=uint8), array([297552,  16510]))
train+validation class ratio: [1191140   64478]


In [10]:
t_v_labels = np.row_stack([train_dataset1.labels, val_dataset1.labels])
print(t_v_labels.shape)
class_counts = np.zeros(t_v_labels.shape[1])
s = 0
for i in range(t_v_labels.shape[1]):
    count = np.unique(t_v_labels[:,i], return_counts=True)[1][1]
    #
    #print(count + np.unique(train_dataset1.labels[:,i],return_counts=True)[1][0])
    class_counts[i] = count
    #print(count)
#train_dataset1.labels[:,0].shape
#train_dataset1.labels.shape
class_counts
#class_counts

(89687, 14)


array([ 9115.,  2190., 10553., 15932.,  4613.,  4988.,  1111.,  4209.,
        3710.,  1890.,  2007.,  1324.,  2651.,   185.])

In [11]:
total_img_wclass = class_counts.sum()

In [12]:
zero_class = train_dataset1.labels[np.all(train_dataset1.labels == 0, axis=1)]

In [13]:
print(f"with classes: {total_img_wclass}")
print(f"without classes: {len(zero_class)}")

with classes: 64478.0
without classes: 42405


As we see above, the classes are imbalanced. However, most classes have at least 1000 images belong to its class except for hernia and pneumonia. Therefore, we will downsample the number of data to total 1000 images associated with the class labels other than hernia and pneumonia.

One thing to note is that some images has 0 in all classes, which may indicate images of healthy patients. So, in fact, there are 15 classes for this problem. Hence, we will downsample these images to total 1000 images

In [36]:
# a) training without data balancing

# resnet18 model
#writer = SummaryWriter("runs/breast")
model_imbalace = ResNet18(pretrained=False, last_activation=None, num_classes=14)
model_imbalace.conv1 = torch.nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

# HyperParameters
SEED = 123
#BATCH_SIZE = 128
BATCH_SIZE = 512
imratio = 0.1 # for demo 
total_epochs = 50
decay_epochs = [50, 75]

lr = 0.1
margin = 1.0
epoch_decay = 0.003 # refers gamma in the paper
weight_decay = 0.0001


# loss and optimizer
loss_fn = AUCMLoss()
optimizer = PESG(model_imbalace, 
                 loss_fn=loss_fn,
                 lr=lr, 
                 momentum=0.9,
                 margin=margin, 
                 epoch_decay=epoch_decay, 
                 weight_decay=weight_decay)




In [37]:
trainloader = data.DataLoader(dataset=train_dataset1, batch_size=BATCH_SIZE, shuffle=True)
valloader = data.DataLoader(dataset=val_dataset1, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
trainloader_eval = data.DataLoader(dataset=train_dataset1, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [38]:
# early stopping patience; how long to wait after last time validation loss improved
patience=7
multilabel_train(model_imbalace, loss_fn, optimizer, total_epochs, trainloader, 
                                      trainloader_eval, valloader, patience=patience, decay_epochs=decay_epochs)

Start Training
------------------------------
torch.Size([512, 1, 28, 28]) torch.Size([512, 14])
torch.Size([512, 1, 28, 28]) torch.Size([512, 14])
torch.Size([512, 1, 28, 28]) torch.Size([512, 14])
torch.Size([512, 1, 28, 28]) torch.Size([512, 14])
torch.Size([512, 1, 28, 28]) torch.Size([512, 14])


KeyboardInterrupt: 

In [15]:
'''
plt.rcParams["figure.figsize"] = (9,5)
x=np.arange(len(train_log))
plt.figure()
plt.plot(x, train_log, linestyle='-', label='Train Set', linewidth=3)
plt.plot(x, val_log,  linestyle='-', label='Validation Set', linewidth=3)
plt.title('AUCMLoss (10% NoduleMNIST)',fontsize=25)
plt.legend(fontsize=15)
plt.ylabel('AUROC', fontsize=25)
plt.xlabel('Epoch', fontsize=25)
'''

NameError: name 'train_log' is not defined

In [16]:
# Testing AUC
test_loader = data.DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE)

score_list = list()
label_list = list()
for _, d in enumerate(test_loader, 0):
    tmp_data, tmp_label = d
    #tmp_data, tmp_label = tmp_data.cuda(), tmp_label.cuda()
    
    tmp_score = model_imbalace(tmp_data).detach().clone().cpu()
    score_list.append(tmp_score)
    label_list.append(tmp_label.cpu())

test_label = torch.cat(label_list)
test_score = torch.cat(score_list)
                   
test_auc = metrics.roc_auc_score(test_label, test_score)                   
print("Test : %.4f"%test_auc, flush=True)

Test : 0.4960


In [ ]:
# save the imbalanced model
torch.save(model_imbalace.state_dict(), "chest/chest_imbalance.pth")

Data Balancing

In [14]:
# Contruct balanced dataset
# 1. downsample images associated with no class to 1000 images
# 2. downsample all the other classes to 1000 images per class by using random sampling
# 3. upsample images have label hernia with data augmentation techniques
combined_data = torch.utils.data.ConcatDataset([train_dataset1, val_dataset1 ])
len(combined_data)

89687

In [15]:
num_imgs = 1000

In [16]:
downsample_bool = class_counts >= num_imgs
print(downsample_bool)
#random_sampler = data.RandomSampler(combined_data, num_samples=num_imgs)

[ True  True  True  True  True  True  True  True  True  True  True  True
  True False]


In [17]:
from math import ceil

# downsample no class data to total 1000 images
combined_imgs = np.row_stack([train_dataset1.imgs, val_dataset1.imgs])
combined_labels = np.row_stack([train_dataset1.labels, val_dataset1.labels])
print(combined_labels[np.all(combined_labels == 0, axis=1)].shape)
desired_class_idx = [i for i in range(len(combined_labels)) if all(combined_labels[i,:] == np.zeros(combined_labels.shape[1]))]
len(desired_class_idx)
filt_imgs = torch.from_numpy(combined_imgs[desired_class_idx, :, :])
filt_labels = torch.from_numpy(combined_labels[desired_class_idx,:])
filt_train = torch.utils.data.TensorDataset(filt_imgs, filt_labels)
random_sampler = data.RandomSampler(filt_train, num_samples=num_imgs)
added_loader = data.DataLoader(filt_train, batch_size=BATCH_SIZE, sampler=random_sampler)
added_diter = iter(added_loader)
examples, labels = next(added_diter)
X, y = examples, labels
for i in range(1, ceil(num_imgs / BATCH_SIZE)):
    examples, labels = next(added_diter)
    X = torch.cat([X, examples])
    y = torch.cat([y, labels])
X.shape, y.shape

(48484, 14)


(torch.Size([1000, 28, 28]), torch.Size([1000, 14]))

In [18]:
from math import ceil
# downsample other classes
combined_imgs = torch.cat([torch.from_numpy(train_dataset1.imgs),
                                   torch.from_numpy(val_dataset1.imgs)])
combined_labels = torch.cat([torch.from_numpy(train_dataset1.labels),
                                     torch.from_numpy(val_dataset1.labels)])
for j in range(len(downsample_bool)):
    desired_class_idx = [i for i in range(len(combined_labels)) if combined_labels[i,j] == 1]
    print(len(desired_class_idx))
    if downsample_bool[j]:
        print(combined_imgs.shape)
        print(combined_labels.shape)
        filt_imgs = combined_imgs[desired_class_idx, :, :]
        filt_labels = combined_labels[desired_class_idx,:]
        print(filt_imgs.shape)
        print(filt_labels.shape)
        filt_train = torch.utils.data.TensorDataset(filt_imgs, filt_labels)
        random_sampler = data.RandomSampler(filt_train, num_samples=num_imgs)
        added_loader = data.DataLoader(filt_train, batch_size=BATCH_SIZE, sampler=random_sampler)
        added_diter = iter(added_loader)
        examples, labels = next(added_diter)
        added_tensors_img, added_tensors_labels = examples, labels
        #for i in range(1, ceil(len(combined_train) / BATCH_SIZE)):
        for i in range(1, ceil(num_imgs / BATCH_SIZE)):
            examples, labels = next(added_diter)
            added_tensors_img = torch.cat([added_tensors_img, examples])
            added_tensors_labels = torch.cat([added_tensors_labels, labels])
    '''
    elif j == 6:
        added_tensors_img = combined_imgs[desired_class_idx, :, :]
        added_tensors_labels = combined_labels[desired_class_idx,:]
        #added_tensors_img = torch.from_numpy(filt_imgs)
        #added_tensors_labels = torch.from_numpy(filt_labels)
        print(added_tensors_img.shape, added_tensors_labels.shape)  
    '''
    X = torch.cat([X, added_tensors_img])
    y = torch.cat([y, added_tensors_labels])
X = X.float()

9115
torch.Size([89687, 28, 28])
torch.Size([89687, 14])
torch.Size([9115, 28, 28])
torch.Size([9115, 14])
2190
torch.Size([89687, 28, 28])
torch.Size([89687, 14])
torch.Size([2190, 28, 28])
torch.Size([2190, 14])
10553
torch.Size([89687, 28, 28])
torch.Size([89687, 14])
torch.Size([10553, 28, 28])
torch.Size([10553, 14])
15932
torch.Size([89687, 28, 28])
torch.Size([89687, 14])
torch.Size([15932, 28, 28])
torch.Size([15932, 14])
4613
torch.Size([89687, 28, 28])
torch.Size([89687, 14])
torch.Size([4613, 28, 28])
torch.Size([4613, 14])
4988
torch.Size([89687, 28, 28])
torch.Size([89687, 14])
torch.Size([4988, 28, 28])
torch.Size([4988, 14])
1111
torch.Size([89687, 28, 28])
torch.Size([89687, 14])
torch.Size([1111, 28, 28])
torch.Size([1111, 14])
4209
torch.Size([89687, 28, 28])
torch.Size([89687, 14])
torch.Size([4209, 28, 28])
torch.Size([4209, 14])
3710
torch.Size([89687, 28, 28])
torch.Size([89687, 14])
torch.Size([3710, 28, 28])
torch.Size([3710, 14])
1890
torch.Size([89687, 28, 28]

In [19]:
'''
added_dataloader = data.DataLoader(new_combined_train, batch_size=BATCH_SIZE, sampler=random_sampler)

added_diter = iter(added_dataloader)
examples, labels = next(added_diter)
added_tensors_img, added_tensors_labels = examples, labels
#for i in range(1, ceil(len(combined_train) / BATCH_SIZE)):
for i in range(1, ceil(count_diff / BATCH_SIZE)):
    examples, labels = next(added_diter)
    added_tensors_img = torch.cat([added_tensors_img, examples])
    added_tensors_labels = torch.cat([added_tensors_labels, labels])
    
X = torch.cat([ct_tensors_img, added_tensors_img])
y = torch.cat([ct_tensors_labels, added_tensors_labels])
'''

'\nadded_dataloader = data.DataLoader(new_combined_train, batch_size=BATCH_SIZE, sampler=random_sampler)\n\nadded_diter = iter(added_dataloader)\nexamples, labels = next(added_diter)\nadded_tensors_img, added_tensors_labels = examples, labels\n#for i in range(1, ceil(len(combined_train) / BATCH_SIZE)):\nfor i in range(1, ceil(count_diff / BATCH_SIZE)):\n    examples, labels = next(added_diter)\n    added_tensors_img = torch.cat([added_tensors_img, examples])\n    added_tensors_labels = torch.cat([added_tensors_labels, labels])\n    \nX = torch.cat([ct_tensors_img, added_tensors_img])\ny = torch.cat([ct_tensors_labels, added_tensors_labels])\n'

In [20]:
# upsampling hernia. Increase # of images from 144 to 1000
# We will overcome the data imbalance issue by upsamling the minority class using data augmentation
# since 1000 is almost eight times as many as the number of images belong to hernia(144), 
# seven data augmentation techniques will be applied to increase the number of images in the minority class

# HorizontalFlip
data_transform2 = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomHorizontalFlip(p=1.0)
])
# Gaussian Noise
data_transform3 = transforms.Compose([
    transforms.ToTensor(),
    AddGaussianNoise(0., 0.2)
])
# Normalize
data_transform4 = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])
# Elastic Transforms
data_transform5 = transforms.Compose([
    transforms.ToTensor(),
    transforms.ElasticTransform()
])
# Vertical Flip
data_transform6 = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomVerticalFlip(p=1.0)
])
# Random rotation between 30 to 90 degree
data_transform7 = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomRotation(degrees=[30,90])
])
# Random rotation between 90 to 150 degree
data_transform8 = transforms.Compose([
    transforms.ToTensor(),
    transforms.RandomRotation(degrees=[90,150])
])


In [21]:
# Data upsampling function
def create_new_img_by_augmentations(desired_class, data_transform):
    # load more training and validation dataset to create new data using data augmentation
    # param: desired_class(list) is a list of classes to be kept,
    # and all the other classes not in this list will be filtered out
    # param: data_transform(pytorch.transforms) is a data transform method to be applied to dataset
    # return: pytorch dataset
    train_dataset = DataClass(split='train', transform=data_transform)
    val_dataset = DataClass(split='val', transform=data_transform)
    combined_imgs = torch.cat([torch.from_numpy(train_dataset.imgs),
                                   torch.from_numpy(val_dataset.imgs)])
    combined_labels = torch.cat([torch.from_numpy(train_dataset.labels),
                                     torch.from_numpy(val_dataset.labels)])
    desired_class_idx = [i for i in range(len(combined_labels)) if combined_labels[i,desired_class] == 1]
    '''
    filtered_trainset = torch.utils.data.Subset(train_dataset,
                            [i for i in range(len(train_dataset)) if train_dataset.labels[i,desired_class] == 1])
    filtered_valset = torch.utils.data.Subset(val_dataset,
                            [i for i in range(len(val_dataset)) if val_dataset.labels[i,desired_class] == 1])
    combined_train = torch.utils.data.ConcatDataset([filtered_trainset, filtered_valset])
    '''
    filt_imgs = combined_imgs[desired_class_idx, :, :]
    filt_labels = combined_labels[desired_class_idx,:]
    print(filt_imgs.shape)
    print(filt_labels.shape)
    combined_train = torch.utils.data.TensorDataset(filt_imgs, filt_labels)
    print(len(combined_train))
    return combined_train

In [22]:
desired_class = 13
new_combined_train = torch.utils.data.ConcatDataset([
    create_new_img_by_augmentations(desired_class, data_transform=data_transform2),
    create_new_img_by_augmentations(desired_class, data_transform=data_transform3),
    create_new_img_by_augmentations(desired_class, data_transform=data_transform4),
    create_new_img_by_augmentations(desired_class, data_transform=data_transform5),
    create_new_img_by_augmentations(desired_class, data_transform=data_transform6),
    create_new_img_by_augmentations(desired_class, data_transform=data_transform7),
    create_new_img_by_augmentations(desired_class, data_transform=data_transform8),
])
len(new_combined_train)

torch.Size([185, 28, 28])
torch.Size([185, 14])
185
torch.Size([185, 28, 28])
torch.Size([185, 14])
185
torch.Size([185, 28, 28])
torch.Size([185, 14])
185
torch.Size([185, 28, 28])
torch.Size([185, 14])
185
torch.Size([185, 28, 28])
torch.Size([185, 14])
185
torch.Size([185, 28, 28])
torch.Size([185, 14])
185
torch.Size([185, 28, 28])
torch.Size([185, 14])
185


1295

In [23]:
BATCH_SIZE = 512
random_sampler = data.RandomSampler(new_combined_train, num_samples=num_imgs)
added_dataloader = data.DataLoader(new_combined_train, batch_size=BATCH_SIZE, sampler=random_sampler)

added_diter = iter(added_dataloader)
examples, labels = next(added_diter)
added_tensors_img, added_tensors_labels = examples, labels
#for i in range(1, ceil(len(combined_train) / BATCH_SIZE)):
for i in range(1, ceil(num_imgs / BATCH_SIZE)):
    examples, labels = next(added_diter)
    print(examples.shape, labels.shape)
    added_tensors_img = torch.cat([added_tensors_img, examples])
    added_tensors_labels = torch.cat([added_tensors_labels, labels])

print(added_tensors_img.shape)
print(X.shape)
X = torch.cat([X, added_tensors_img])
y = torch.cat([y, added_tensors_labels])
X = X.float()

print(X.shape[0])
y.unique(return_counts=True)

torch.Size([488, 28, 28]) torch.Size([488, 14])
torch.Size([1000, 28, 28])
torch.Size([15000, 28, 28])
16000


(tensor([0, 1], dtype=torch.uint8), tensor([193252,  30748]))

In [24]:
for i in range(y.shape[1]):
    count = np.unique(y[:,i], return_counts=True)[1][1]
    #
    #print(count + np.unique(train_dataset1.labels[:,i],return_counts=True)[1][0])
    class_counts[i] = count
    #print(count)
#train_dataset1.labels[:,0].shape
#train_dataset1.labels.shape
#print(int(class_counts.sum()), X.shape[0])
class_counts
#class_counts

array([3207., 1426., 4001., 4509., 2214., 2029., 1348., 1950., 1934.,
       1580., 1510., 1305., 2705., 1030.])

In [25]:
X.shape

torch.Size([16000, 28, 28])

In [26]:
X = X.reshape((X.shape[0],1,X.shape[1],X.shape[2]))

Not perfectly balanced, but the newly constructed data set is more balanced compared to the originial dataset.

In [27]:
dataset = torch.utils.data.TensorDataset(X, y)
val_d, train_d = torch.utils.data.random_split(dataset, lengths=[0.2,0.8])
len(val_d), len(train_d)

(3200, 12800)

In [28]:
'''
# splitting train and validation data with balanced ratio
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=7)
len(X_train)
'''

'\n# splitting train and validation data with balanced ratio\nfrom sklearn.model_selection import train_test_split\nX_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=7)\nlen(X_train)\n'

In [29]:
'''
train_d = torch.utils.data.TensorDataset(X_train, y_train)
val_d = torch.utils.data.TensorDataset(X_val, y_val)
'''

'\ntrain_d = torch.utils.data.TensorDataset(X_train, y_train)\nval_d = torch.utils.data.TensorDataset(X_val, y_val)\n'

Step 3 & 4:
 
 
3. tune optimizer, batch size, and step size and Hyperparameter tuning.
 
 
4. Lastly, model training and testing with the best dataset

In [33]:
# resnet18 model
from libauc.models import resnet18 as ResNet18

SEED = 123
imratio = 0.1 # for demo 
total_epochs = 50
decay_epochs = [50, 75]


margin = 1.0
epoch_decay = 0.003 # refers gamma in the paper
weight_decay = 0.0001

# HyperParameters
optmzr = ['adam', 'PESG']
BATCH_SIZE = [512, 1024]
lr = [1e-3, 1e-1]

max_auc = 0
best_hyperparams = []  # optimizer, batch_size, lr order
best_model = None
# hyperparameter tuning based on gridsearch
for i in range(len(optmzr)*len(BATCH_SIZE)*len(lr)):
    model_i = ResNet18(pretrained=False, last_activation=None, num_classes=14)
    model_i.conv1 = torch.nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    
    lr_idx = i % len(lr)
    batch_idx = (i // len(lr)) % len(BATCH_SIZE)
    optmzr_idx = i // (len(lr) * len(BATCH_SIZE))
    
    trainloader = data.DataLoader(dataset=train_d, batch_size=BATCH_SIZE[batch_idx], shuffle=True)
    valloader = data.DataLoader(dataset=val_d, batch_size=BATCH_SIZE[batch_idx], shuffle=False, num_workers=2)
    trainloader_eval = data.DataLoader(train_d, batch_size=BATCH_SIZE[batch_idx], shuffle=False, num_workers=2)
    

    # loss and optimizer
    loss_fn = AUCMLoss()

    if optmzr[optmzr_idx] == 'adam':  # use adam optimizer
        #optimizer = torch.optim.Adam(model_i.parameters(), lr=lr[lr_idx], weight_decay=weight_decay)
        optimizer = Adam(model_i,
                 lr=lr[lr_idx],
                 weight_decay=weight_decay)
    elif optmzr[optmzr_idx] == 'momentum':  # use momentous SGD optimzer
        #optimizer = optim.SGD(model_i.parameters(), lr=lr[lr_idx], momentum=0.9)
        optimizer = SGD(model_i, 
                 lr=lr[lr_idx],
                 momentum=0.9,
                 weight_decay=weight_decay)
    else:  # PESG
        optimizer = PESG(model_i, 
                 loss_fn=loss_fn,
                 lr=lr[lr_idx], 
                 momentum=0.9,
                 #margin=margin)
                 margin=margin, 
                 epoch_decay=epoch_decay, 
                 weight_decay=weight_decay)
    
    print(f"optimzer: {optmzr[optmzr_idx]}, step size: {lr[lr_idx]}, batch size: {BATCH_SIZE[batch_idx]}")
    multilabel_train(model_i, loss_fn, optimizer, total_epochs, trainloader, trainloader_eval, valloader,
          patience=7, decay_epochs=decay_epochs)

    # evaluate the performance on validation set to select the best hyperparameters
    val_loader = data.DataLoader(dataset=val_d, batch_size=BATCH_SIZE[batch_idx], num_workers=2)
    score_list = list()
    label_list = list()
    for _, d in enumerate(val_loader, 0):
        tmp_data, tmp_label = d
        #tmp_data, tmp_label = tmp_data.cuda(), tmp_label.cuda()

        tmp_score = model_i(tmp_data).detach().clone().cpu()
        #print(tmp_score)
        score_list.append(tmp_score)
        label_list.append(tmp_label.cpu())

    val_label = torch.cat(label_list)
    val_score = torch.cat(score_list)

    val_auc = metrics.roc_auc_score(val_label, val_score)                   
    print("Validation : %.4f"%val_auc, flush=True)
                        #val_auc??
    if val_auc > max_auc:
        #val_auc??
        max_auc = val_auc #val_auc?
        best_hyperparams = [optmzr[optmzr_idx], lr[lr_idx], BATCH_SIZE[batch_idx]]
        best_model = model_i

optimzer: adam, step size: 0.001, batch size: 512
Start Training
------------------------------
Epoch=0, BatchID=0, Val_AUC=0.5163
0
Validation : 0.4918
optimzer: adam, step size: 0.1, batch size: 512
Start Training
------------------------------
Epoch=0, BatchID=0, Val_AUC=0.5000


KeyboardInterrupt: 

In [35]:
batch_size = best_hyperparams[2]
#X_test = torch.from_numpy(test_dataset.imgs)
#X_test = X_test.float()
#test_d = torch.utils.data.TensorDataset(X_test, torch.from_numpy(test_dataset.labels))

#test_loader = data.DataLoader(dataset=test_d, batch_size=batch_size)
test_loader = data.DataLoader(dataset=test_dataset, batch_size=batch_size)

# Testing AUC
score_list = list()
label_list = list()
for _, d in enumerate(test_loader, 0):
    tmp_data, tmp_label = d
    #tmp_data, tmp_label = tmp_data.cuda(), tmp_label.cuda()
    
    tmp_score = best_model(tmp_data).detach().clone().cpu()
    score_list.append(tmp_score)
    label_list.append(tmp_label.cpu())

test_label = torch.cat(label_list)
test_score = torch.cat(score_list)
                   
test_auc = metrics.roc_auc_score(test_label, test_score)                   
print("Test : %.4f"%test_auc, flush=True)

Test : 0.4708


In [ ]:
# save the best model
torch.save(best_model.state_dict(), "chest/chest_hyperparams.pth")